In [4]:
import gc
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ==========================================
# CONFIGURATIE
# ==========================================
DATA_PATH = "transformed_data_sampled.csv"
TARGET_COL = "status"

# ==========================================
# 1. DATA INLADEN
# ==========================================
print("1. Data inladen...")

sample = pd.read_csv(DATA_PATH, nrows=100)

dtypes = {}
for col in sample.columns:
    if sample[col].dtype == "float64":
        dtypes[col] = "float32"
    elif sample[col].dtype == "int64":
        dtypes[col] = "int32"
    else:
        dtypes[col] = sample[col].dtype

df = pd.read_csv(DATA_PATH, dtype=dtypes)

# ==========================================
# 2. STATUSCODES GROEPEREN (Aangepast naar SUCCESS vs ERROR)
# ==========================================
print("2. Statuscodes groeperen...")

# Originele labelencoder mapping:
# 0=200, 1=201, 2=204, 3=400, 4=401, 5=404, 6=500

status_group_mapping = {
    0: "SUCCESS",        # 200
    1: "SUCCESS",        # 201
    2: "SUCCESS",        # 204

    3: "ERROR",          # 400 (Client)
    4: "ERROR",          # 401 (Client)
    5: "ERROR",          # 404 (Client)

    6: "ERROR"           # 500 (Server -> Nu samengevoegd met Client)
}

df["status_grouped"] = df[TARGET_COL].map(status_group_mapping)

print("\nNieuwe verdeling:")
print(df["status_grouped"].value_counts())

# Nieuwe labels encoden
le = LabelEncoder()
df[TARGET_COL] = le.fit_transform(df["status_grouped"])

print("\nNieuwe encoded labels:")
for i, cls in enumerate(le.classes_):
    print(f"{i} -> {cls}")

# tijdelijke kolom verwijderen
df.drop(columns=["status_grouped"], inplace=True)

# ==========================================
# 3. X / y SPLIT
# ==========================================
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

del df
gc.collect()

# ==========================================
# 4. TRAIN / VALIDATION SPLIT
# ==========================================
print("\n3. Dataset opsplitsen...")

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

num_classes = len(np.unique(y_train))
print(f"Aantal klassen: {num_classes} (Binair)")

del X, y
gc.collect()

# ==========================================
# 5. CLASS WEIGHTS
# ==========================================
print("\n4. Class weights berekenen...")

class_counts = y_train.value_counts().sort_index()

class_weights = {
    cls: len(y_train) / (len(class_counts) * count)
    for cls, count in class_counts.items()
}

print(class_counts)

sample_weights = y_train.map(class_weights).values

# ==========================================
# 6. DMATRIX
# ==========================================
dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    weight=sample_weights
)

dval = xgb.DMatrix(
    X_val,
    label=y_val
)

# ==========================================
# 7. PARAMETERS (Geoptimaliseerd voor Binair)
# ==========================================
params = {
    # Aangepast van multi:softprob naar binary:logistic omdat we nu 2 klassen hebben
    "objective": "binary:logistic", 

    "tree_method": "hist",

    "max_depth": 6,
    "min_child_weight": 5,

    "learning_rate": 0.05,

    # Aangepast van mlogloss naar logloss voor binaire classificatie
    "eval_metric": "logloss", 

    "subsample": 0.7,
    "colsample_bytree": 0.7,

    "max_delta_step": 1,

    "verbosity": 1
}

# 'num_class' is niet meer nodig en verwijderd bij binary:logistic

# ==========================================
# 8. TRAINEN
# ==========================================
print("\n5. Start training...")

evallist = [(dval, "validation"), (dtrain, "train")]

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=evallist,            # <-- Hier stond eerst alleen 'evallist', 'evals=' lost het op!
    early_stopping_rounds=50,
    verbose_eval=50
)
# ==========================================
# 9. EVALUATIE
# ==========================================
print("\n=== EVALUATIE ===")

best_iteration = bst.best_iteration

preds_prob = bst.predict(
    dval,
    iteration_range=(0, best_iteration + 1)
)

# Bij binary:logistic geeft predict() een 1D array van kansen voor klasse 1 (SUCCESS).
# We ronden af: >= 0.5 wordt klasse 1, < 0.5 wordt klasse 0.
y_pred = (preds_prob >= 0.5).astype(int)

print("\nClassificatierapport:")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

# ==========================================
# 10. LABELS TONEN
# ==========================================
print("\nLabel mapping:")
for i, cls in enumerate(le.classes_):
    print(f"{i} -> {cls}")

1. Data inladen...
2. Statuscodes groeperen...

Nieuwe verdeling:
status_grouped
ERROR      103812
SUCCESS     46188
Name: count, dtype: int64

Nieuwe encoded labels:
0 -> ERROR
1 -> SUCCESS

3. Dataset opsplitsen...


/tmp/ipykernel_1267889/3183913955.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["status_grouped"] = df[TARGET_COL].map(status_group_mapping)


Aantal klassen: 2 (Binair)

4. Class weights berekenen...
status
0    83050
1    36950
Name: count, dtype: int64

5. Start training...
[0]	validation-logloss:0.68423	train-logloss:0.68194
[50]	validation-logloss:0.51115	train-logloss:0.48596
[100]	validation-logloss:0.48904	train-logloss:0.46408
[150]	validation-logloss:0.48261	train-logloss:0.45764
[200]	validation-logloss:0.47780	train-logloss:0.45290
[250]	validation-logloss:0.47379	train-logloss:0.44858
[300]	validation-logloss:0.47033	train-logloss:0.44549
[350]	validation-logloss:0.46806	train-logloss:0.44297
[400]	validation-logloss:0.46580	train-logloss:0.44081
[450]	validation-logloss:0.46474	train-logloss:0.43927
[500]	validation-logloss:0.46337	train-logloss:0.43788
[550]	validation-logloss:0.46236	train-logloss:0.43676
[600]	validation-logloss:0.46132	train-logloss:0.43573
[650]	validation-logloss:0.46066	train-logloss:0.43473
[700]	validation-logloss:0.46005	train-logloss:0.43405
[750]	validation-logloss:0.45914	train-logl

In [5]:
# Train een mini-model met 5 bomen om de belangrijkste feature te vinden
mini_bst = xgb.train(params, dtrain, num_boost_round=5)
importance = mini_bst.get_score(importance_type="gain")
print("Belangrijkste features:", sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3])


Belangrijkste features: [('type_GET', 18986.970703125), ('body_MISSING', 900.24169921875), ('runid', 146.27378845214844)]
